# Orchestration Strategies for Multi-Agent Systems

## Overview

Orchestration is the art of coordinating multiple agents to accomplish complex tasks efficiently. This notebook explores different orchestration strategies and when to use them.

### Topics:

1. Sequential Orchestration
2. Parallel Orchestration
3. Hierarchical Orchestration
4. Dynamic Orchestration
5. Hybrid Approaches

In [ ]:
# Setup
import sys
sys.path.append('..')

from typing import Dict, List, Any, Optional, Callable
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
import time
import asyncio
from concurrent.futures import ThreadPoolExecutor, as_completed

from utils.visualization import (
    visualize_orchestration_pattern,
    create_agent_timeline,
    visualize_agent_graph
)

print("✓ Setup complete!")

## 1. Sequential Orchestration

Agents execute one after another in a predefined order. Output from one agent becomes input to the next.

### Characteristics:
- ✓ Simple to understand and debug
- ✓ Predictable execution flow
- ✗ Slow for independent tasks
- ✗ Bottleneck if one agent is slow

### Use cases:
- Pipeline processing (ETL)
- Multi-stage refinement
- When output depends on previous step

In [ ]:
# Visualize sequential pattern
visualize_orchestration_pattern(
    pattern_type='sequential',
    agents=['Collect', 'Clean', 'Transform', 'Analyze', 'Report']
)

In [ ]:
@dataclass
class Task:
    """Represents a task with input and output."""
    name: str
    input_data: Any
    output_data: Any = None
    status: str = "pending"  # pending, running, completed, failed
    start_time: Optional[float] = None
    end_time: Optional[float] = None
    
    @property
    def duration(self) -> Optional[float]:
        if self.start_time and self.end_time:
            return self.end_time - self.start_time
        return None


class SequentialOrchestrator:
    """Orchestrates agents in sequential order."""
    
    def __init__(self, agents: List[Callable]):
        self.agents = agents
        self.execution_log: List[Task] = []
        
    def execute(self, initial_input: Any) -> Any:
        """Execute agents sequentially."""
        current_data = initial_input
        
        print("🔄 Starting sequential execution...\n")
        
        for i, agent in enumerate(self.agents):
            task = Task(name=f"Agent_{i}_{agent.__name__}", input_data=current_data)
            task.status = "running"
            task.start_time = time.time()
            
            print(f"▶️  Step {i+1}/{len(self.agents)}: {agent.__name__}")
            print(f"   Input: {str(current_data)[:60]}...")
            
            try:
                current_data = agent(current_data)
                task.output_data = current_data
                task.status = "completed"
                print(f"   Output: {str(current_data)[:60]}...")
            except Exception as e:
                task.status = "failed"
                print(f"   ❌ Error: {e}")
                raise
            finally:
                task.end_time = time.time()
                self.execution_log.append(task)
                print(f"   ⏱️  Duration: {task.duration:.3f}s\n")
        
        print(f"✅ Sequential execution complete!")
        return current_data


# Example: Data processing pipeline
def collect_data(input_data):
    """Simulates data collection."""
    time.sleep(0.3)
    return {"raw_data": [1, 2, 3, 4, 5], "source": "api"}

def clean_data(input_data):
    """Simulates data cleaning."""
    time.sleep(0.2)
    data = input_data.get("raw_data", [])
    return {"cleaned_data": [x for x in data if x > 0], "source": input_data["source"]}

def transform_data(input_data):
    """Simulates data transformation."""
    time.sleep(0.25)
    data = input_data.get("cleaned_data", [])
    return {"transformed_data": [x * 2 for x in data]}

def analyze_data(input_data):
    """Simulates data analysis."""
    time.sleep(0.2)
    data = input_data.get("transformed_data", [])
    return {"analysis": {"mean": sum(data) / len(data), "count": len(data)}}


# Execute pipeline
orchestrator = SequentialOrchestrator([
    collect_data,
    clean_data,
    transform_data,
    analyze_data
])

result = orchestrator.execute({})
print(f"\n📊 Final result: {result}")
print(f"📈 Total execution time: {sum(t.duration for t in orchestrator.execution_log):.3f}s")

## 2. Parallel Orchestration

Multiple agents execute simultaneously on independent tasks, with results aggregated at the end.

### Characteristics:
- ✓ Fast for independent tasks
- ✓ Better resource utilization
- ✗ Complex error handling
- ✗ Requires task independence

### Use cases:
- Distributed computation
- Parallel data processing
- Multiple independent queries

In [ ]:
# Visualize parallel pattern
visualize_orchestration_pattern(
    pattern_type='parallel',
    agents=['Researcher', 'Coder', 'Tester', 'Documenter']
)

In [ ]:
class ParallelOrchestrator:
    """Orchestrates agents in parallel."""
    
    def __init__(self, agents: List[Callable], max_workers: int = 4):
        self.agents = agents
        self.max_workers = max_workers
        self.execution_log: List[Task] = []
        
    def execute(self, inputs: List[Any]) -> List[Any]:
        """Execute agents in parallel."""
        print(f"🚀 Starting parallel execution with {len(self.agents)} agents...\n")
        
        results = []
        
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            # Submit all tasks
            future_to_agent = {}
            for i, (agent, input_data) in enumerate(zip(self.agents, inputs)):
                task = Task(name=f"{agent.__name__}", input_data=input_data)
                task.start_time = time.time()
                task.status = "running"
                
                future = executor.submit(agent, input_data)
                future_to_agent[future] = (agent, task)
                print(f"▶️  Launched: {agent.__name__}")
            
            print(f"\n⏳ Waiting for agents to complete...\n")
            
            # Collect results as they complete
            for future in as_completed(future_to_agent):
                agent, task = future_to_agent[future]
                
                try:
                    result = future.result()
                    task.output_data = result
                    task.status = "completed"
                    task.end_time = time.time()
                    
                    results.append(result)
                    print(f"✅ {agent.__name__} completed ({task.duration:.3f}s)")
                    print(f"   Output: {str(result)[:60]}...")
                except Exception as e:
                    task.status = "failed"
                    task.end_time = time.time()
                    print(f"❌ {agent.__name__} failed: {e}")
                finally:
                    self.execution_log.append(task)
        
        print(f"\n✅ Parallel execution complete!")
        return results


# Example: Parallel research tasks
def research_papers(query):
    """Simulates paper research."""
    time.sleep(0.5)
    return {"papers": [f"Paper on {query}"], "count": 1}

def research_code(query):
    """Simulates code research."""
    time.sleep(0.4)
    return {"repos": [f"Repo for {query}"], "stars": 100}

def research_datasets(query):
    """Simulates dataset research."""
    time.sleep(0.3)
    return {"datasets": [f"Dataset for {query}"], "size": "1GB"}

def research_tools(query):
    """Simulates tool research."""
    time.sleep(0.6)
    return {"tools": [f"Tool for {query}"], "version": "1.0"}


# Execute in parallel
parallel_orchestrator = ParallelOrchestrator([
    research_papers,
    research_code,
    research_datasets,
    research_tools
])

topic = "multi-agent systems"
parallel_results = parallel_orchestrator.execute([topic, topic, topic, topic])

print(f"\n📊 All results collected: {len(parallel_results)} items")
max_time = max(t.duration for t in parallel_orchestrator.execution_log)
print(f"📈 Total execution time: {max_time:.3f}s (parallelized)")

## 3. Hierarchical Orchestration

A supervisor agent delegates tasks to manager agents, who coordinate worker agents.

### Characteristics:
- ✓ Clear authority structure
- ✓ Scalable for large systems
- ✓ Good for complex tasks
- ✗ Communication overhead
- ✗ More complex to implement

### Use cases:
- Large-scale projects
- Multi-team coordination
- Enterprise workflows

In [ ]:
# Visualize hierarchical pattern
visualize_orchestration_pattern(
    pattern_type='hierarchical',
    agents=['Supervisor', 'FrontendManager', 'BackendManager', 'UIWorker', 'APIWorker', 'DBWorker']
)

In [ ]:
from typing import Protocol

class Agent(Protocol):
    """Protocol for agent interface."""
    def execute(self, task: Dict[str, Any]) -> Dict[str, Any]:
        ...


class WorkerAgent:
    """Base worker agent."""
    
    def __init__(self, name: str, skill: str):
        self.name = name
        self.skill = skill
        
    def execute(self, task: Dict[str, Any]) -> Dict[str, Any]:
        print(f"  👷 [{self.name}] executing: {task['description']}")
        time.sleep(0.1)
        return {
            "status": "completed",
            "result": f"{self.skill} task completed",
            "agent": self.name
        }


class ManagerAgent:
    """Manager agent that coordinates workers."""
    
    def __init__(self, name: str, workers: List[WorkerAgent]):
        self.name = name
        self.workers = workers
        
    def delegate(self, tasks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        print(f"\n👔 [{self.name}] delegating {len(tasks)} tasks to {len(self.workers)} workers")
        
        results = []
        for task, worker in zip(tasks, self.workers):
            result = worker.execute(task)
            results.append(result)
            
        return results


class SupervisorAgent:
    """Top-level supervisor coordinating managers."""
    
    def __init__(self, name: str, managers: List[ManagerAgent]):
        self.name = name
        self.managers = managers
        
    def orchestrate(self, project: Dict[str, Any]) -> Dict[str, Any]:
        print(f"\n🎯 [{self.name}] starting project: {project['name']}")
        
        all_results = []
        
        for manager, tasks in zip(self.managers, project['task_groups']):
            results = manager.delegate(tasks)
            all_results.extend(results)
            
        print(f"\n✅ [{self.name}] project complete: {len(all_results)} tasks finished")
        
        return {
            "project": project['name'],
            "status": "completed",
            "tasks_completed": len(all_results),
            "results": all_results
        }


# Build hierarchical system
# Workers
ui_designer = WorkerAgent("UIDesigner", "UI Design")
frontend_dev = WorkerAgent("FrontendDev", "Frontend Development")
api_dev = WorkerAgent("APIDev", "API Development")
db_admin = WorkerAgent("DBAdmin", "Database Management")

# Managers
frontend_manager = ManagerAgent("FrontendManager", [ui_designer, frontend_dev])
backend_manager = ManagerAgent("BackendManager", [api_dev, db_admin])

# Supervisor
project_supervisor = SupervisorAgent("ProjectSupervisor", [frontend_manager, backend_manager])

# Execute hierarchical orchestration
project = {
    "name": "E-commerce Platform",
    "task_groups": [
        [
            {"description": "Design product page UI", "priority": "high"},
            {"description": "Implement shopping cart", "priority": "high"}
        ],
        [
            {"description": "Build product API", "priority": "high"},
            {"description": "Setup product database", "priority": "high"}
        ]
    ]
}

final_result = project_supervisor.orchestrate(project)
print(f"\n📊 Final status: {final_result['status']}")
print(f"📈 Tasks completed: {final_result['tasks_completed']}")

## 4. Comparison: Sequential vs Parallel vs Hierarchical

Let's compare execution times for the same set of tasks:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simulated execution times
patterns = ['Sequential', 'Parallel', 'Hierarchical']
execution_times = [1.0, 0.6, 0.7]  # Relative times
communication_overhead = [0.05, 0.1, 0.15]
complexity = [1, 3, 4]  # Implementation complexity (1-5 scale)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Execution time
axes[0].bar(patterns, execution_times, color=['#3498db', '#2ecc71', '#e74c3c'])
axes[0].set_ylabel('Relative Time')
axes[0].set_title('Execution Time Comparison')
axes[0].set_ylim([0, 1.2])

# Communication overhead
axes[1].bar(patterns, communication_overhead, color=['#3498db', '#2ecc71', '#e74c3c'])
axes[1].set_ylabel('Overhead')
axes[1].set_title('Communication Overhead')
axes[1].set_ylim([0, 0.2])

# Implementation complexity
axes[2].bar(patterns, complexity, color=['#3498db', '#2ecc71', '#e74c3c'])
axes[2].set_ylabel('Complexity (1-5)')
axes[2].set_title('Implementation Complexity')
axes[2].set_ylim([0, 5])

plt.tight_layout()
plt.show()

## 5. Dynamic Orchestration

Agents are selected and coordinated based on runtime conditions and task requirements.

In [ ]:
class DynamicOrchestrator:
    """Orchestrator that adapts based on task requirements."""
    
    def __init__(self, agent_registry: Dict[str, Callable]):
        self.agent_registry = agent_registry
        
    def select_strategy(self, task: Dict[str, Any]) -> str:
        """Dynamically select orchestration strategy."""
        if task.get('requires_sequence'):
            return 'sequential'
        elif task.get('subtasks_independent'):
            return 'parallel'
        elif task.get('complexity') == 'high':
            return 'hierarchical'
        else:
            return 'sequential'  # Default
            
    def select_agents(self, task: Dict[str, Any]) -> List[str]:
        """Select agents based on required capabilities."""
        required_skills = task.get('required_skills', [])
        selected = []
        
        for skill in required_skills:
            if skill in self.agent_registry:
                selected.append(skill)
                
        return selected
        
    def execute(self, task: Dict[str, Any]) -> Dict[str, Any]:
        """Execute task with dynamic strategy selection."""
        print(f"🤖 Analyzing task: {task['name']}")
        
        strategy = self.select_strategy(task)
        agents = self.select_agents(task)
        
        print(f"   Strategy: {strategy}")
        print(f"   Selected agents: {agents}")
        
        # Execute based on strategy
        results = []
        for agent_name in agents:
            agent = self.agent_registry[agent_name]
            result = agent(task)
            results.append(result)
            
        return {
            "task": task['name'],
            "strategy": strategy,
            "agents_used": agents,
            "results": results
        }


# Example agents
def research_agent(task):
    return {"research": "findings for " + task['name']}

def coding_agent(task):
    return {"code": "implementation for " + task['name']}

def testing_agent(task):
    return {"tests": "test suite for " + task['name']}


# Setup dynamic orchestrator
registry = {
    'research': research_agent,
    'coding': coding_agent,
    'testing': testing_agent
}

dynamic_orch = DynamicOrchestrator(registry)

# Execute different types of tasks
tasks = [
    {
        "name": "Build ML Pipeline",
        "required_skills": ['research', 'coding'],
        "requires_sequence": True
    },
    {
        "name": "Test Multiple Components",
        "required_skills": ['testing'],
        "subtasks_independent": True
    }
]

for task in tasks:
    result = dynamic_orch.execute(task)
    print(f"   ✅ Completed: {result['task']}\n")

## Decision Matrix: Choosing an Orchestration Strategy

| Factor | Sequential | Parallel | Hierarchical | Dynamic |
|--------|-----------|----------|--------------|----------|
| **Task Dependencies** | Strong | None | Mixed | Variable |
| **Execution Speed** | Slow | Fast | Medium | Variable |
| **Complexity** | Low | Medium | High | High |
| **Scalability** | Limited | Good | Excellent | Good |
| **Debugging** | Easy | Medium | Hard | Hard |
| **Resource Usage** | Low | High | Medium | Variable |

### When to use each:

**Sequential**: Pipeline tasks, dependencies between steps, simple workflows

**Parallel**: Independent tasks, time-critical operations, embarrassingly parallel problems

**Hierarchical**: Large systems, clear responsibility structure, enterprise workflows

**Dynamic**: Varying task types, adaptive systems, unpredictable workloads

## Exercise: Design Your Orchestration Strategy

Consider a system for automated code review:

**Agents needed:**
- Syntax checker
- Security scanner
- Performance analyzer
- Style checker
- Documentation validator

**Questions:**
1. Which orchestration pattern would you use?
2. Can any tasks run in parallel?
3. Are there any dependencies?
4. How would you aggregate results?

In [ ]:
# Your orchestration design here

# Hint: Most code review tasks are independent and could benefit from parallel execution
# Consider how you would aggregate results and determine overall pass/fail

## Summary

In this notebook, we covered:

- ✓ Sequential orchestration for dependent tasks
- ✓ Parallel orchestration for independent tasks
- ✓ Hierarchical orchestration for complex systems
- ✓ Dynamic orchestration for adaptive systems
- ✓ How to choose the right strategy

**Next:** In Notebook 4, we'll build a hands-on multi-agent system using LangGraph, bringing all these concepts together!